In [ ]:
import fitz
import json
import os
import pandas as pd

1. Konversi PDF to JSON

In [ ]:
class BytesEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, bytes):
            return obj.decode('utf-16')
        return json.JSONEncoder.default(self, obj)

In [ ]:
from pathlib import Path
# directory = "[Full] Daftar Kamus Ekstraksi" 
# directory_hasil = "[Full] Bentuk JSON"

directory = '/Users/rafaeldewandaru/SEMESTER 8/Tugas Akhir/PDF'
directory_hasil = "/Users/rafaeldewandaru/SEMESTER 8/Tugas Akhir/TAEkstraksiKamus/[Full] Bentuk JSON"

pdfgagal = []

for filename in os.listdir(directory):
    print("=== Processing File " + filename + " ===")
    try:
        with open(directory + "/" + filename,"rb") as f:
            document = fitz.open(f)
        
        new_filename = os.path.splitext(filename)[0]
        with open(directory_hasil + "/" + new_filename + "-hasil.json", "w",encoding="utf-8") as t:
            counter = 0
            result = {}
            for page in document:
                data = {}
                value = page.get_text('dict')
                for i in value['blocks']:
                    try:
                        i.pop('image')
                    except:
                        continue
                data["page"] = counter
                data["text"] = value
                result[str(counter)] = data
                counter += 1
            json.dump(result,t)
        
        f.close()
        t.close()
    except:
        print("=== File Gagal " + filename + " ===")
        pdfgagal.append(filename)

In [ ]:
for i in pdfgagal:
    print(i)

2. Ekstrak Informasi JSON

In [ ]:
import pandas as pd
import numpy as np
import os
import re
import json as js
from pathlib import Path
from tqdm import tqdm

In [ ]:
# ganti directory untuk menjalankan program

# directory_kamus = "/Users/rafaeldewandaru/SEMESTER 8/Tugas Akhir/TAEkstraksiKamus/[Full] Daftar Kamus Ekstraksi"
directory_JSON_raw = "/Users/rafaeldewandaru/SEMESTER 8/Tugas Akhir/TAEkstraksiKamus/[Full] Bentuk JSON"
directory_hasil = "/Users/rafaeldewandaru/SEMESTER 8/Tugas Akhir/TAEkstraksiKamus/[Full] Raw CSV JSON all information"

list_json_raw = os.listdir(directory_JSON_raw)
# list_directory_kamus = os.listdir(directory_kamus)

print(list_json_raw)

In [ ]:
def extract_information_json(data):
    result = {
        "text":[],
        "size":[],
        "font":[],
        "x0":[], #[0]
        "y0":[], #[1]
        "x1":[], #[2]
        "y1":[], #[3]
        "page":[]
    }
    
    lines = []
    line_pages = []
    for key in data:
        page = data.get(key)
        for block in page.get("text").get("blocks"):
            if block.get("lines") is not None:
                lines.append(block.get("lines"))
                line_pages.append(page.get("page")) 
    
    spans = []
    span_pages = []
    for i in range(len(lines)):
        ln = lines[i]
        for detail_ln in ln:
            if detail_ln.get("spans") is not None:
                spans.append(detail_ln.get("spans"))
                span_pages.append(line_pages[i])
    
    for i in range(len(spans)):
        span = spans[i]
        for detail in span:
            result.get("text").append(detail.get("text"))
            result.get("size").append(detail.get("size"))
            result.get("font").append(detail.get("font"))
            result.get("x0").append(round(detail.get("bbox")[0],2))
            result.get("y0").append(round(detail.get("bbox")[1],2))
            result.get("x1").append(round(detail.get("bbox")[2],2))
            result.get("y1").append(round(detail.get("bbox")[3],2))
            result.get("page").append(span_pages[i]+1)
    
    return result

def is_contain_only_whitespaces(s):
    if re.match(r'^\s*$', str(s)): return True
    return False

def clean_information(data):
    result = {
        "text":[],
        "size":[],
        "font":[],
        "x0":[], #[0]
        "y0":[], #[1]
        "x1":[], #[2]
        "y1":[], #[3]
        "page":[]
    }
    
    i = 0
    while i < len(data["text"]):
        text = data["text"][i]
        if is_contain_only_whitespaces(text): # skip kata yang hanya mengandung whitespace
            i += 1
        else:
            result["text"].append(data["text"][i].strip())
            result["size"].append(data["size"][i])
            result["font"].append(data["font"][i])
            result["x0"].append(data["x0"][i])
            result["y0"].append(data["y0"][i])
            result["x1"].append(data["x1"][i])
            result["y1"].append(data["y1"][i])
            result["page"].append(data["page"][i])
            i += 1
            
    return result

def seperate_span(data): # tokenisasi span berdasarkan spasi
    result = {
        "text":[],
        "size":[],
        "font":[],
        "x0":[], #[0]
        "y0":[], #[1]
        "x1":[], #[2]
        "y1":[], #[3]
        "page":[]
    }
    
    i = 0
    while i < len(data["text"]):
        curr_font = data["font"][i].lower()
        list_txt = data["text"][i].strip().split(" ")
        
        selisih_x = data["x1"][i] - data["x0"][i] # x1 - x0
        satuan_x = selisih_x/len(data["text"][i])
                
        if len(list_txt)==1: # jika span ternyata hanya satu kata
            result["text"].append(data["text"][i].strip())
            result["size"].append(data["size"][i])
            result["font"].append(data["font"][i])
            result["x0"].append(data["x0"][i])
            result["y0"].append(data["y0"][i])
            result["x1"].append(data["x1"][i])
            result["y1"].append(data["y1"][i])
            result["page"].append(data["page"][i])
            i+=1
        else:
            x0 = data["x0"][i]
            x1 = data["x1"][i] - len(data["text"][i])*satuan_x
            x1 = round(x1,2)

            y0 = data["y0"][i] # koordinat y tetap
            y1 = data["y1"][i]

            for txt in list_txt:
                x1 += len(txt) * satuan_x
                x1 = round(x1,2)

                result["text"].append(txt.strip())
                result["size"].append(data["size"][i])
                result["font"].append(data["font"][i])
                result["x0"].append(x0)
                result["y0"].append(y0)
                result["x1"].append(x1)
                result["y1"].append(y1)
                result["page"].append(data["page"][i])

                x0 += (len(txt)+1) * satuan_x
                x0 = round(x0,2)
            i+=1
            
        
    return result

def is_contain_bold_and_italic(font):
    contains_bold = False; contains_italic = False
    for i in font:
        if "bold" in i.lower(): contains_bold = True
        if "italic" in i.lower(): contains_italic = True
        if contains_bold == True and contains_italic == True: return True
    return False

def is_contain_bold(font):
    contains_bold = False
    for i in font:
        if "bold" in i.lower(): contains_bold = True
    return contains_bold

def count_bold(font):
    cnt = 0
    for i in font:
        if "bold" in i.lower(): 
            cnt += 1
    return cnt

In [ ]:
daftar_kamus_layak = []

daftar_kamus = os.listdir(directory_JSON_raw)
daftar_kamus = sorted(daftar_kamus)
for filename in tqdm(daftar_kamus):
    print("====" + filename + "====")
    try:
        with open(directory_JSON_raw + "/" + filename,"rb") as f:
            data = js.load(f)
        f.close()
        
        new_filename = os.path.splitext(filename)[0]
        result_raw = extract_information_json(data)
        result_clean = clean_information(result_raw)
        result = seperate_span(result_clean)
        
        if (is_contain_bold_and_italic(result["font"])):
            csv_res = pd.DataFrame.from_dict(result)
            csv_res.to_csv(directory_hasil + "/" + new_filename + "-ekstraksi.csv",index=False)
            daftar_kamus_layak.append(new_filename)
    except:
         continue
    
print("Jumlah kamus yang layak:", len(daftar_kamus_layak))

In [ ]:
split_kamus = pd.read_csv("Split Kamus.csv")

kamus_split = split_kamus["Kamus"].values.tolist()

awal = split_kamus["halaman_pertama"].values.tolist()

akhir = split_kamus["halaman_terakhir"].values.tolist()

In [ ]:
daftar_kamus_tidak_layak = []
daftar_kamus_layak_setelah_displit = []

directory_hasil_raw = os.listdir(directory_hasil)

for i in range(len(directory_hasil_raw)):
    filename_clean = directory_hasil_raw[i]
    # print("====" + filename_clean + "====" + str(kamus_split[i]))
    print("====" + filename_clean + "====")
    
    kamus = pd.read_csv(directory_hasil + "/" + filename_clean)
    kamus = kamus[kamus["page"] >= awal[i]]
    kamus = kamus[kamus["page"] <= akhir[i]]
    kamus = kamus.reset_index(drop=True)
    
    if is_contain_bold(kamus["font"].values.tolist()):
        print("==== Memenuhi Kriteria ====")
        directory_hasil_clean = "[Full] CSV JSON all information"
        kamus.to_csv(directory_hasil_clean + "/" + filename_clean,index=False)
        daftar_kamus_layak_setelah_displit.append(filename_clean)
    else:
        print("==== Tidak Memenuhi Kriteria ====")
        daftar_kamus_tidak_layak.append(filename_clean)
    

In [ ]:
print("Jumlah kamus yang tidak layak setelah displit:", len(daftar_kamus_tidak_layak))
for i in daftar_kamus_tidak_layak:
    print(i)
    
print("====================================")

print("Jumlah kamus yang layak setelah displit:", len(daftar_kamus_layak_setelah_displit))    
for i in daftar_kamus_layak_setelah_displit:
    print(i)

In [ ]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm

In [ ]:
directory = "[Full] CSV JSON all information"

In [ ]:
def count_font_bold(data):
    cnt = 0
    for i in data:
        if "bold" in i.lower():
            cnt += 1
    return cnt

def count_font_italic(data):
    cnt = 0
    for i in data:
        if "italic" in i.lower():
            cnt += 1
    return cnt

In [ ]:
page = [1,1,1,1,2,2,2,2,]

page = np.array(page)
page = np.unique(page)
page[-1]

result = {
    "nama_kamus":[],
    "jumlah_bold":[],
    "jumlah_italic":[],
    "mean_ukuran_char":[],
    "banyak_halaman":[]
}

for filename in tqdm(os.listdir(directory)):
    data = pd.read_csv(directory + "/" + filename)
    
    sum_bold = count_font_bold(data["font"].values.tolist())
    sum_italic = count_font_italic(data["font"].values.tolist())
    
    size = data["size"].values.tolist()

    page = np.array(data["page"].values.tolist())
    page = np.unique(page)
    
    result['nama_kamus'].append(filename)
    result['jumlah_bold'].append(sum_bold)
    result['jumlah_italic'].append(sum_italic)
    result['mean_ukuran_char'].append(round(np.mean(size),2))
    result['banyak_halaman'].append(len(page))

pandas_res = pd.DataFrame.from_dict(result)
# tampilkan seluruh baris dan seluruh nilai pada kolom
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

display(pandas_res)

# reset option
pd.reset_option("display")

pandas_res.to_csv(directory + "/exploratory_kamus_analysis.csv",index=False)

pandas_res = pd.read_csv(directory + "/exploratory_kamus_analysis.csv")

pandas_res = pandas_res[(pandas_res["jumlah_bold"] >= pandas_res["banyak_halaman"]) & (pandas_res["jumlah_italic"] >= pandas_res["banyak_halaman"])]
pandas_res = pandas_res.reset_index(drop=True)
# tampilkan seluruh baris dan seluruh nilai pada kolom
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
pandas_res.to_csv("daftar_kamus_ekstraks.csv",index=False)

display(pandas_res)

# reset option
pd.reset_option("display")

pandas_res = pd.read_csv(directory + "/exploratory_kamus_analysis.csv")

pandas_res = pandas_res[(pandas_res["jumlah_bold"] < pandas_res["banyak_halaman"]) | (pandas_res["jumlah_italic"] < pandas_res["banyak_halaman"])]
pandas_res = pandas_res.reset_index(drop=True)
# tampilkan seluruh baris dan seluruh nilai pada kolom
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

display(pandas_res)

# reset option
pd.reset_option("display")
pandas_res.to_csv("daftar kamus yang tidak diekstrak.csv",index=False)

4. One Entry Corpus Code Font Approach

In [ ]:
import pandas as pd
import numpy as np
import os
import re
import json as js
from pathlib import Path
from tqdm import tqdm
from ast import literal_eval

In [ ]:
directory_kamus = "Daftar Kamus Analisis Machine Readable"
directory_kamus_full = "[Full] Daftar Kamus Ekstraksi"

In [ ]:
POS = ["v","a","n","pron","adv","num","p"]

In [ ]:
# Algoritma Tambahan
def is_contain_bold_and_italic(font):
    contains_bold = False; contains_italic = False
    for i in font:
        if "bold" in i.lower(): contains_bold = True
        if "italic" in i.lower(): contains_italic = True
        if contains_bold == True and contains_italic == True: return True
    return False

def is_last_fonem(s): # baru dapat handle fonem (/../) dan ([...])
    if re.match(r'^.*\]$',str(s)): return True
    if re.match(r'^.*\/$',str(s)): return True
    return False

def is_start_fonem(s): # baru dapat handle fonem (/../) dan ([...])
    if re.match(r'^\[.*',str(s)): return True
    if re.match(r'^\/.*',str(s)): return True
    return False

def is_bold_contains_POS(s):
    kata = s.strip()
    
    if len(kata) > 1:
        if is_contain_only_whitespaces(kata[-2]) and (kata[-1] in POS): return True
    else:
        if (kata[-1] in POS): return True
    
    return False

def is_contain_only_whitespaces(s):
    if re.match(r'^\s*$', str(s)): return True
    return False

def is_end_entri(s):
    symbol = [";",",",":"]
    if s in symbol:
        return True
    else:
        return False
    
# make entry by font
def make_entry_by_font(data):
    result = {
        "Entri":[],
        "entry_font_size_pos":[],
        "posisi_entry":[],
        "page":[]
    }
    
    entry = []; entry_with_font_size_pos = []; pos_dummy = None; page_dummy = None
    
    for ind in data.index:
        txt = data["text"][ind]
        size = data["size"][ind]
        size = round(size,2)
        fnt = data["font"][ind].lower()
        x0 = round(data["x0"][ind],2)
        y0 = round(data["y0"][ind],2)
        x1 = round(data["x1"][ind],2)
        y1 = round(data["y1"][ind],2)
        pos = [x0,y0,x1,y1]
        page = data["page"][ind]
        
        if "bold" in fnt and entry == []: # start entry
            entry.append(txt)
            entry_with_font_size_pos.append([txt,fnt,size,[x0,y0,x1,y1]])
            pos_dummy = pos
            page_dummy = page
            
        elif "bold" in fnt and entry != []:
            prev_txt = data["text"][ind-1].strip()
            prev_fnt = data["font"][ind-1].lower()
            entry_result = " ".join(entry).strip()
            
            if "bold" in prev_fnt and not txt[0].isdigit() and is_bold_contains_POS(entry_result): # handle prakategorial tanpa koma
                result["Entri"].append(entry_result)
                result["entry_font_size_pos"].append(entry_with_font_size_pos)
                result["posisi_entry"].append(pos_dummy)
                
                if page == page_dummy: 
                    result["page"].append(page_dummy)
                else:
                    result["page"].append([page_dummy,page])
                    
                entry = []; entry_with_font_size_pos = []; pos_dummy = None; page_dummy = None
                entry.append(txt) # mulai entry baru
                entry_with_font_size_pos.append([txt,fnt,size,[x0,y0,x1,y1]])
                pos_dummy = pos
                page_dummy = page
                
            elif "bold" in prev_fnt and (txt[0].isdigit() or not is_bold_contains_POS(entry_result)): # handle kata bold yang terpisah
                entry.append(txt) 
                entry_with_font_size_pos.append([txt,fnt,size,[x0,y0,x1,y1]])
                
            elif "bold" not in prev_fnt and (txt[0].isdigit() or is_start_fonem(txt)): # polisemi dan fonem bold
                entry.append(txt) 
                entry_with_font_size_pos.append([txt,fnt,size,[x0,y0,x1,y1]])
                
            else: 
                result["Entri"].append(entry_result)
                result["entry_font_size_pos"].append(entry_with_font_size_pos)
                result["posisi_entry"].append(pos_dummy)
                
                if page == page_dummy: 
                    result["page"].append(page_dummy)
                else:
                    result["page"].append([page_dummy,page])
                    
                entry = []; entry_with_font_size_pos = []; pos_dummy = None; page_dummy = None
                entry.append(txt) # mulai entry baru
                entry_with_font_size_pos.append([txt,fnt,size,[x0,y0,x1,y1]])
                pos_dummy = pos
                page_dummy = page
                
        elif "bold" not in fnt.lower() and entry != []:
            entry.append(txt) 
            entry_with_font_size_pos.append([txt,fnt,size,[x0,y0,x1,y1]])
            
        else:
            result["Entri"].append(txt.strip())
            result["entry_font_size_pos"].append([[txt,fnt,size,[x0,y0,x1,y1]]])
            result["posisi_entry"].append(pos)
            result["page"].append(page)
            

    if entry != []: # jika ada entry yang tertinggal
        entry_result = " ".join(entry).strip()
        result["Entri"].append(entry_result)
        result["entry_font_size_pos"].append(entry_with_font_size_pos)
        result["posisi_entry"].append(pos_dummy)
        result["page"].append(page_dummy)

    return result

# algoritma bersihkan entry dari fonem
def clean_entry(data):
    result = {
        "Entri":[],
        "entry_font_size_pos":[],
        "posisi_entry":[],
        "page":[]
    }
    
    for i in range(len(data["Entri"])): # remove fonem
        txt = data["Entri"][i] # data text
        
        if not is_contain_only_whitespaces(txt):
            
            entry_font_size_pos = data["entry_font_size_pos"][i]
            txt = re.sub(r'\[.*?\]',"",txt)
            entry_font_size_pos = clean_entry_font_size_paranthesis(entry_font_size_pos)

            txt = re.sub(r'\/.*?\/',"",txt)
            entry_font_size_pos = clean_entry_font_size_slash(entry_font_size_pos)

            clean = re.sub(' +', ' ', txt) ## remove multiple whitespace
            result["Entri"].append(clean.strip())
            result["entry_font_size_pos"].append(entry_font_size_pos)

            result['posisi_entry'].append(data['posisi_entry'][i])
            result['page'].append(data['page'][i])
    
    for j in range(1,len(result['Entri'])): # fix symbol
        array_simbol = []; array_simbol_font_size_pos = []
        
        prev_txt_split = result["Entri"][j-1].split(" ")
        prev_entri_font_size_pos = result['entry_font_size_pos'][j-1]
        
        # buang seluruh simbol, kecuali ; pada entri sebelumnya
        while (prev_txt_split[-1] != "") and (not is_end_entri(prev_txt_split[-1][-1])):
            if (prev_txt_split[-1][0].isalnum()) or (prev_txt_split[-1][-1].isalnum()): 
                break
                
            else:
                if (prev_txt_split==[] or prev_entri_font_size_pos == []):break
                
                array_simbol.append(prev_txt_split[-1])
                array_simbol_font_size_pos.append(prev_entri_font_size_pos[-1])
                del prev_txt_split[-1]
                del prev_entri_font_size_pos[-1]
                
                result["Entri"][j-1] = " ".join(prev_txt_split)
                result['entry_font_size_pos'][j-1] = prev_entri_font_size_pos
            
            if (prev_txt_split==[] or prev_entri_font_size_pos == []):break
        
        txt_split = result['Entri'][j].split(" ")
        if is_end_entri(txt_split[0]): 
            result['Entri'][j-1] = result['Entri'][j-1] + txt_split[0]
            result['entry_font_size_pos'][j-1].append(result['entry_font_size_pos'][j][0])
            
            del txt_split[0]
            result['entry_font_size_pos'][j] = result['entry_font_size_pos'][j][1:]
            result['Entri'][j] = " ".join(txt_split)
        
        if array_simbol != []:
            new_entry = []
            new_entry.extend(array_simbol)
            new_entry.extend(txt_split)
            result['Entri'][j] = " ".join(new_entry)
            
            new_entry_font_size_pos = []
            new_entry_font_size_pos.extend(array_simbol_font_size_pos)
            new_entry_font_size_pos.extend(result['entry_font_size_pos'][j])
            result['entry_font_size_pos'][j] = new_entry_font_size_pos    
    
    for l in range(len(result['entry_font_size_pos'])):
        if result['entry_font_size_pos'][l] != []:
            result['posisi_entry'][l] = result['entry_font_size_pos'][l][0][-1]
        
    return result

def clean_entry_font_size_paranthesis(data):
    clean_data = []
    i = 0
    
    while i < len(data):
        txt = data[i][0]
        if re.match(r'^.*\[.*?\].*$',str(txt)): ## kasus ...[..]...
            clean = re.sub(r'\[.*?\]',"",txt)
            if clean == "":
                i += 1
            else:
                clean_data.append([clean.strip(),data[i][1],data[i][2],data[i][3]])
                i += 1
        elif re.match(r'^.*\[.*',str(txt)): ## kasus ...[...
            nxt = i+1
            if nxt > len(data)-1: # i di indeks terakhir
                clean_data.append(data[i])
                break
                
            nxt_txt = data[nxt][0]
            while not re.match(r'^.*\].*$',str(nxt_txt)): # mencari "...]...."
                nxt += 1
                if nxt > len(data)-1: break
                nxt_txt = data[nxt][0]
            
            if nxt > len(data)-1: # jika "....]..." tidak ditemukan
                for k in range(i,nxt):
                    clean_data.append(data[k])
                break
            else:
                ## append [ pertama
                clean = txt.split("[",1)[0]
                if clean != "":
                    clean_data.append([clean.strip(),data[i][1],data[i][2],data[i][3]])
                    
                ## append ] kedua
                clean_nxt = nxt_txt.split("]",1)[1]
                if clean_nxt != "":
                    clean_data.append([clean_nxt.strip(),data[nxt][1],data[nxt][2],data[i][3]])
                
                i = nxt+1
        else:
            clean_data.append(data[i])
            i += 1
    
    return clean_data


def clean_entry_font_size_slash(data):
    clean_data = []
    i = 0
    
    while i < len(data):
        txt = data[i][0]
        if re.match(r'^.*\/.*?\/.*$',str(txt)): ## kasus .../../...
            clean = re.sub(r'\/.*?\/',"",txt)
            if clean == "":
                i += 1
            else:
                clean_data.append([clean.strip(),data[i][1],data[i][2],data[i][3]])
                i += 1
        elif re.match(r'^.*\/.*',str(txt)): ## kasus .../...
            nxt = i+1
            if nxt > len(data)-1: # i di indeks terakhir
                clean_data.append(data[i])
                break
                
            nxt_txt = data[nxt][0]
            while not re.match(r'^.*\/.*$',str(nxt_txt)): # mencari ".../...."
                nxt += 1
                if nxt > len(data)-1: break
                nxt_txt = data[nxt][0]
            
            if nxt > len(data)-1: # jika "..../..." tidak ditemukan
                for k in range(i,nxt):
                    clean_data.append(data[k])
                break
            else:
                ## append / pertama
                clean = txt.split("/",1)[0]
                if clean != "":
                    clean_data.append([clean.strip(),data[i][1],data[i][2],data[i][3]])
                    
                ## append / kedua
                clean_nxt = nxt_txt.split("/",1)[1]
                if clean_nxt != "":
                    clean_data.append([clean_nxt.strip(),data[nxt][1],data[nxt][2],data[i][3]])
                
                i = nxt+1
        else:
            clean_data.append(data[i])
            i += 1
    
    return clean_data

def fix_page(pages):
    clean_page = []
    cnt = 1
    
    for i in range(len(pages)):
        if i == 0:
            clean_page.append(cnt)
        else:
            if isinstance(pages[i], list):
                clean_page.append([cnt,cnt+1])
                cnt += 1
            else:
                if isinstance(pages[i-1], list):
                    clean_page.append(cnt)
                else:
                    if pages[i] == pages[i-1]:
                        clean_page.append(cnt)
                    else:
                        cnt += 1
                        clean_page.append(cnt)
    return clean_page

# memisahkan prakategorial
def seperate_prakategorial(data):
    result = {
        "Entri":[],
        "entry_font_size_pos":[],
        "posisi_entry":[],
        "page":[]
    }
    
    for i in range(len(data["Entri"])):
        txt = data["Entri"][i]
        split_txt = txt.strip().split(",",1)
        
        if len(split_txt) < 2 or txt[-1] == ",": # tidak terdapat koma atau koma berada di akhir
            result['Entri'].append(txt)
            result['entry_font_size_pos'].append(data['entry_font_size_pos'][i])
            result['page'].append(data['page'][i])
            result['posisi_entry'].append(data['posisi_entry'][i])
        
        else:
            frst_entri = split_txt[0].strip().split(" ")
            sec_entri = split_txt[1].strip().split(" ")
            
            for j in range(len(frst_entri)):
                frst_entri[j] = frst_entri[j].strip()
            
            for k in range(len(sec_entri)):
                sec_entri[k] = sec_entri[k].strip()
                
            if len(frst_entri) <= 2 and (frst_entri[0] == "" or frst_entri[0] == ","): # koma berada di awal entri
                result['Entri'].append(txt)
                result['entry_font_size_pos'].append(data['entry_font_size_pos'][i])
                result['page'].append(data['page'][i])
                result['posisi_entry'].append(data['posisi_entry'][i])
            
            else:
                inf_frst_entri = data['entry_font_size_pos'][i][:len(frst_entri)]
                
                if "bold" in inf_frst_entri[-1][1].lower() or frst_entri[-1] in POS:
                    if (len(frst_entri) + len(sec_entri)) == len(data['entry_font_size_pos'][i]): # kasus koma menempel
                        frst_entri[-1] = frst_entri[-1] + ","
                        inf_sec_entri = data['entry_font_size_pos'][i][len(frst_entri):]

                    else: # kasus koma tidak menempel
                        frst_entri.append(",")
                        inf_frst_entri = data['entry_font_size_pos'][i][:len(frst_entri)+1]
                        inf_sec_entri = data['entry_font_size_pos'][i][len(frst_entri)+1:]
                        
                    # entri pertama
                    result['Entri'].append(" ".join(frst_entri))
                    result['entry_font_size_pos'].append(inf_frst_entri)
                    result['page'].append(data['page'][i])
                    result['posisi_entry'].append(data['posisi_entry'][i])

                    # entri kedua
                    result['Entri'].append(" ".join(sec_entri))
                    result['entry_font_size_pos'].append(inf_sec_entri)
                    result['page'].append(data['page'][i])
                    result['posisi_entry'].append(data['posisi_entry'][i])

                else:
                    result['Entri'].append(txt)
                    result['entry_font_size_pos'].append(data['entry_font_size_pos'][i])
                    result['page'].append(data['page'][i])
                    result['posisi_entry'].append(data['posisi_entry'][i])
                
                
    return result

def categorize_prakategorial(entries):
    output = []
    
    for i in entries:
        txt_split = i.split(" ")
        if i == "" or len(i)==1:
            output.append(0)
        else:
            if re.match(r'.*\,$',str(i)) and len(txt_split) <= 3: 
                output.append(1)
            elif is_contain_only_whitespaces(i[-2]) and (i[-1] in POS):
                output.append(1)
            else:
                output.append(0)
    return output

# pisahkan main entry atau entry-entry pokok yang masih tergabung
def seperate_joined_entry(data, kategori):
    result_entries = []
    result_entries_with_font_size_pos = []
    result_posisi = []
    result_page = []
    
    entry = []
    entry_with_font_size_pos = []
    posisi_dummy = None;
    
    i = 0
    while i < len(data["Entri"]):
        
        if len(data["entry_font_size_pos"][i]) < 2: # jika hanya terdiri dari 1 kata atau 0 kata
            result_entries.append(data["Entri"][i])
            result_entries_with_font_size_pos.append(data["entry_font_size_pos"][i])
            result_posisi.append(data["posisi_entry"][i])
            result_page.append(data["page"][i])
            entry = []; entry_with_font_size_pos = []; posisi_dummy = None
        
        else:
            posisi = data["posisi_entry"][i]
            batas_atas = round(posisi[0] + (posisi[0] * (1/100)),2)

            detail = data["entry_font_size_pos"][i][0] # handle index 0

            entry.append(detail[0].strip())
            entry_with_font_size_pos.append(detail)
            posisi_dummy = detail[-1]

            if (kategori[i] == 0): 
                result_entries.append(data["Entri"][i])
                result_entries_with_font_size_pos.append(data["entry_font_size_pos"][i])
                result_posisi.append(data["posisi_entry"][i])
                result_page.append(data["page"][i])
                entry = []; entry_with_font_size_pos = []; posisi_dummy = None

            else: # pisahkan entri pokok yang tergabung
                batas_bawah = round(posisi[0] - (posisi[0] * (1/100)),2) # error 1%

                for j in range(1,len(data["entry_font_size_pos"][i])):
                    detail = data["entry_font_size_pos"][i][j]; 
                    posisi_x0 = round(float(detail[-1][0]))

                    if batas_bawah <= posisi_x0 <= batas_atas: # pisahkan entry
                        joined_entry = (" ").join(entry)
                        result_entries.append(joined_entry)
                        result_entries_with_font_size_pos.append(entry_with_font_size_pos)
                        result_posisi.append(posisi_dummy)
                        result_page.append(data["page"][i])

                        entry = []; entry_with_font_size_pos = []
                        entry.append(detail[0].strip())
                        entry_with_font_size_pos.append(detail)
                        posisi_dummy = detail[-1]

                    else:
                        entry.append(detail[0].strip())
                        entry_with_font_size_pos.append(detail)
        
        if entry != []:
            joined_entry = (" ").join(entry)
            result_entries.append(joined_entry)
            result_entries_with_font_size_pos.append(entry_with_font_size_pos)
            result_posisi.append(posisi_dummy)
            result_page.append(data["page"][i])
            entry = []; entry_with_font_size_pos = []; posisi_dummy = None

        i += 1
    
    return {
        "Entri":result_entries,
        "entry_font_size_pos":result_entries_with_font_size_pos,
        "posisi_entry":result_posisi,
        "page":result_page
    }

# Algoritma join seperate entry
def join_seperate_entry(data):
    result_entry = []
    result_entry_with_font_size_pos = []
    result_posisi = []
    result_page = []
    is_padanan_lema = []
    
    i = 0
    while i < len(data["Entri"]):
        if i == (len(entries)-1): 
            result_entry.append(data["Entri"][i])
            result_entry_with_font_size_pos.append(data["entry_font_size_pos"][i])
            result_posisi.append(data["posisi_entry"][i])
            result_page.append(data["page"][i])
            is_padanan_lema.append(data["is_padanan_lema"][i])
            break
        
        joined_entry = data["Entri"][i]
        joined_entry_with_font_size_pos = data["entry_font_size_pos"][i]
        curr_posisi = data["posisi_entry"][i]
        curr_page = data["page"][i]
        curr_pl = data["is_padanan_lema"][i]
        
        curr_y0 = joined_entry_with_font_size_pos[-1][-1][1]
        batas_atas = curr_y0 + (curr_y0*(1/100)) # error 1%
        batas_bawah = curr_y0 - (curr_y0*(1/100)) # error 1%
        
        
        nxt= i+1
        if nxt > len(entries)-1: 
            result_entry.append(joined_entry)
            result_entry_with_font_size_pos.append(joined_entry_with_font_size_pos)
            result_posisi.append(curr_posisi)
            result_page.append(curr_page)
            is_padanan_lema.append(curr_pl)
            break
            
        else:
            nxt_y0 = round(data["entry_font_size_pos"][nxt][0][-1][1],2)
            while batas_bawah <= nxt_y0 <= batas_atas:
                if curr_pl == 1: break

                joined_entry = joined_entry + " " + data["Entri"][nxt]
                joined_entry_with_font_size_pos.extend(data["entry_font_size_pos"][nxt])

                if isinstance(data["page"][nxt],list):
                    curr_page = data["page"][nxt]
                elif isinstance(curr_page,list):
                    curr_page = curr_page
                elif curr_page != page[nxt]:
                    curr_page = [curr_page, data["page"][nxt]]

                nxt += 1
                
                if nxt == (len(entries)): break
                    
                nxt_y0 = round(data["entry_font_size_pos"][nxt][0][-1][1],2)

                curr_y0 = joined_entry_with_font_size_pos[-1][-1][1]
                curr_pl = data["is_padanan_lema"][nxt]
                batas_atas = curr_y0 + (curr_y0*(1/100)) # error 1%
                batas_bawah = curr_y0 - (curr_y0*(1/100)) # error 1%

            result_entry.append(joined_entry)
            result_entry_with_font_size_pos.append(joined_entry_with_font_size_pos)
            result_posisi.append(curr_posisi)
            result_page.append(curr_page)
            is_padanan_lema.append(curr_pl)
            i = nxt
            
    return {
        "Entri":result_entry,
        "entry_font_size_pos":result_entry_with_font_size_pos,
        "posisi_entry":result_posisi,
        "page":result_page,
        "is_padanan_lema":is_padanan_lema
    }

# kategorisasi entry, untuk memisahkan mana yang main entry dan sub entry
def categorize_main_entry(posisi, pages, lema):
    output = []
    
    i = 0; j = 0
    while i < len(pages):
        if isinstance(pages[i], list): # kasus entry cross page
            prev_posisi_x0 = posisi[i-1][0]
            
            if abs(posisi[i][0] - prev_posisi_x0) <= 3:
                output.append(output[i-1])
                
            else:
                batas_atas = round(prev_posisi_x0 + (prev_posisi_x0 * (2/100)),2) # error 2%
                batas_kolom = 2*batas_atas
                
                if posisi[i][0] > batas_atas and posisi[i][0] < batas_kolom:
                    output.append(0)
                    
                else:
                    output.append(1)
                    
            i += 1; j += 1
        
        else:   
            posisi_by_page = []; lema_by_page = []; curr_page = pages[j] 
        
            while curr_page == pages[i]: # kelompokkan entri berdasarkan halaman
                posisi_by_page.append(posisi[j][0])
                lema_by_page.append(lema[j])
                
                j += 1
                
                if j > len(pages) - 1: break
                    
                curr_page = pages[j]
                
            sorted_posisi = sorted(posisi_by_page) # urutkan
            
            i = j; k = 0; l = 0 # update nilai i
            while k < len(posisi_by_page):
                if lema_by_page[k] == 1:
                    if k == len(posisi_by_page)-1:
                        output.append(1); break
                    else:
                        output.append(1)
                        output.append(0)
                        k += 2
                else:
                    if abs(posisi_by_page[k] - sorted_posisi[l]) > 3:
                        output.append(0); k += 1 # jika tidak sesuai urutan

                    else:
                        output.append(1) # index pertama setelah header atau nomor halaman
                        batas_atas = round(posisi_by_page[k] + (posisi_by_page[k] * (2/100)),2) # error 2%
                        batas_kolom = 2*batas_atas

                        m = k + 1
                        if m > len(posisi_by_page) - 1: break

                        nxt_posisi = posisi_by_page[m]
                        while nxt_posisi > batas_atas and nxt_posisi < batas_kolom:
                            output.append(0); m += 1

                            if m > len(posisi_by_page) - 1: 
                                break 

                            nxt_posisi = posisi_by_page[m]

                        k = m
                        if nxt_posisi < batas_kolom:
                            l += 1
                        else:
                            l = m
                
                
    return output 

def build_corpus_one_entry_by_font(data):
    # tahapan awal, pendekatan dengan font
    result = make_entry_by_font(data)
    clean_result = clean_entry(result)
    clean_result = seperate_prakategorial(clean_result)
    clean_result["is_padanan_lema"] = categorize_prakategorial(clean_result["Entri"])
    clean_result["page"] = fix_page(clean_result["page"])
    
    return clean_result

def build_corpus_one_entry_by_font_pos(data):
    # tahapan awal, pendekatan dengan font
    result = join_seperate_entry(data)
    kategorisasi = categorize_main_entry(result["posisi_entry"], result["page"], result["is_padanan_lema"])
    result = seperate_joined_entry(result, kategorisasi)
#   result = fix_cross_page_entry(result)
    
    clean_result = clean_entry(result)
    clean_result = seperate_prakategorial(result)
    clean_result["is_padanan_lema"] = categorize_prakategorial(clean_result["Entri"])
    return clean_result



In [ ]:
# ganti directory
# directory_CSV = "CSV JSON all information"
directory_CSV = "/Users/rafaeldewandaru/SEMESTER 8/Tugas Akhir/TAEkstraksiKamus/[Full] Raw CSV JSON all information"
directory_hasil = "/Users/rafaeldewandaru/SEMESTER 8/Tugas Akhir/TAEkstraksiKamus/CSV One Entry JSON With Font Approach"

for filename in tqdm(os.listdir(directory_CSV)):
    data = pd.read_csv(directory_CSV + "/" + filename)
    data = data.dropna()
    data = data.reset_index(drop=True)
    input_fonts = data["font"].values.tolist()
    new_filename = os.path.splitext(filename)[0]
    
    if is_contain_bold_and_italic(input_fonts):
        print("====" + new_filename + "====")
        CSV_res = build_corpus_one_entry_by_font(data)

        result_csv = pd.DataFrame.from_dict(CSV_res)
        result_csv = result_csv.reset_index(drop=True)
        
        result_csv = result_csv.dropna()
        result_csv = result_csv.reset_index(drop=True)
        
        result_csv.to_csv(directory_hasil + "/" + new_filename + "-one_entry_from_JSON-font.csv",index=False)

# directory_hasil = "CSV One Entry JSON With Font Approach"

directory_hasil = "/Users/rafaeldewandaru/SEMESTER 8/Tugas Akhir/TAEkstraksiKamus/CSV One Entry JSON With Font Approach"

# drop null data 
for filename in tqdm(os.listdir(directory_hasil)):
    data_clean = pd.read_csv(directory_hasil + "/" + filename)
    data_clean = data_clean.dropna()
    data_clean = data_clean[data_clean["entry_font_size_pos"] != "[]"]
    data_clean = data_clean.reset_index(drop=True)
    
    data_clean.to_csv(directory_hasil + "/" + filename,index=False)

In [ ]:
import ast, re, numpy as np

def safe_parse(s):
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return None
    if isinstance(s, (list, dict, tuple, int, float)):
        return s
    try:
        return ast.literal_eval(s)
    except Exception:
        try:
            # restricted eval allowing common numpy/array constructors
            return eval(s, {'__builtins__': None}, {'np': np, 'array': np.array, 'float': float, 'int': int})
        except Exception:
            nums = re.findall(r'-?\d+\.?\d*', str(s))
            if not nums:
                return s
            if len(nums) == 1:
                return float(nums[0])
            return [float(x) for x in nums]

5. One Entry Corpus Code Font + Posisi Approach

In [ ]:
directory_CSV = "/Users/rafaeldewandaru/SEMESTER 8/Tugas Akhir/TAEkstraksiKamus/CSV One Entry JSON With Font Approach"
directory_hasil = "/Users/rafaeldewandaru/SEMESTER 8/Tugas Akhir/TAEkstraksiKamus/CSV One Entry JSON With Font + Posisi Approach"

for filename in tqdm(os.listdir(directory_CSV)):
    print("====" + filename + "====")
    kamus = pd.read_csv(directory_CSV + "/" + filename)
    kamus = kamus.dropna()
    kamus = kamus.reset_index(drop=True)
    entries = kamus["Entri"].values.tolist()

    entries_font_size_pos = [safe_parse(i) for i in kamus["entry_font_size_pos"].tolist()]
    posisi_entry = [safe_parse(i) for i in kamus["posisi_entry"].tolist()]

    page = []
    for i in kamus["page"].tolist():
        if isinstance(i, (int, np.integer)):
            page.append(int(i))
        else:
            parsed = safe_parse(i)
            try:
                page.append(int(parsed))
            except Exception:
                page.append(parsed)


    # entries_font_size_pos = []
    # for i in kamus["entry_font_size_pos"].values.tolist():
    #     entries_font_size_pos.append(literal_eval(i))

    # posisi_entry = []
    # for i in kamus["posisi_entry"].values.tolist():
    #     posisi_entry.append(literal_eval(i))

    # page = []
    # for i in kamus["page"].values.tolist():
    #     if not isinstance(i,int):
    #         page.append(literal_eval(i))
    #     else:
    #         page.append(int(i))

    input_data = {
        "Entri":entries,
        "entry_font_size_pos":entries_font_size_pos,
        "posisi_entry":posisi_entry,
        "page":page,
        "is_padanan_lema":kamus["is_padanan_lema"].values.tolist()
    }
    
    CSV_res = build_corpus_one_entry_by_font_pos(input_data)

    result_csv = pd.DataFrame.from_dict(CSV_res)
    result_csv = result_csv[result_csv["Entri"] != ""]
    result_csv = result_csv.reset_index(drop=True)

    result_csv = result_csv.dropna()
    result_csv = result_csv.reset_index(drop=True)
    
    new_filename = os.path.splitext(filename)[0]
    result_csv.to_csv(directory_hasil + "/" + new_filename + "-posisi.csv",index=False)